<a href="https://colab.research.google.com/github/Shashwat-006/Learning_PyTorch/blob/main/Learning_PyTorch_Basics_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Theory**

 **Dataset class** in an abstract class which is essentially a blueprint. This is used for deciding how data is loaded & returned when you make a custom class.

It defines:
*   __init__() : tells you how data should be loaded
*   __len__() : returns the total number of samples
*   __getitem__(index) : returns data (and label) at the given index





**DataLoader class** wraps a dataset and handles batching,shuffling & parallel loading.

Dataloader control flow:
*   at start of each epoch, (if shuffle=true) shuffles indices (using a sampler)
*   divides indices into chunks of batch_size
*   for each index in the chunk, data samples are fetched from the dataset object
*   The samples are then collected & combined into a batch (using collate_fn)
*   The batch is returned to the main training loop

# **Making Dataset**

In [1]:
from sklearn.datasets import make_classification
import torch

In [2]:
#creating a synthetic dataset using sklearn
X, y = make_classification(
    n_samples=10,       # Number of samples
    n_features = 2,     # Number of features
    n_informative = 2,  # Number of informative features
    n_redundant = 0,    # Number of redundant features
    n_classes = 2,      # Number of classes
    random_state = 42
)

In [4]:
X.shape

(10, 2)

In [5]:
y.shape

(10,)

In [6]:
#Convert data to PyTorch Tensors
X = torch.tensor(X, dtype = torch.float32)
y = torch.tensor(y, dtype = torch.long)

# **Dataset**

In [8]:
from torch.utils.data import Dataset, DataLoader

In [23]:
class CustomDataset(Dataset):

  def __init__(self, features, labels):
    self.features = features
    self.labels = labels

  def __len__(self):
    return self.features.shape[0]

  def __getitem__(self, index):
    #transformations can be done here (for e.g., resizeing, converting to b/w for images & for textual data as)
    return self.features[index], self.labels[index]

In [16]:
dataset = CustomDataset(X, y)

In [17]:
len(dataset)

10

In [19]:
dataset[5]

(tensor([-0.9382, -0.5430]), tensor(1))

# **DataLoader**

In [25]:
dataloader = DataLoader(dataset, batch_size = 2, shuffle = True)  # there's a num_worker parameter which is used for assigning workers for parallelizing the process

In [22]:
for batch_features, batch_labels in dataloader:
  print(batch_features)
  print(batch_labels)
  print('='*50)

tensor([[-0.9382, -0.5430],
        [ 1.7774,  1.5116]])
tensor([1, 1])
tensor([[ 1.0683, -0.9701],
        [-0.5872, -1.9717]])
tensor([1, 0])
tensor([[ 1.7273, -1.1858],
        [ 1.8997,  0.8344]])
tensor([1, 1])
tensor([[-2.8954,  1.9769],
        [-1.9629, -0.9923]])
tensor([0, 0])
tensor([[-0.7206, -0.9606],
        [-1.1402, -0.8388]])
tensor([0, 0])


The **collate function** in DataLoader specifies how to combine a list of samples from a dataset into a single batch. By default, the DataLoader uses a simple batch collate mechanism, but collate_fn allows us to customize hwo data should be processed & batched

Some important parameters in DataLoader:
*   dataset (mandatory) : Dataset from which DataLoader will pull data
*   batch_size : how many samples to load per batch
*   shuffle : if true, dataset indices will get shuffled
*   num_workers : The number of worker processes used to load data in parallel
*   pin_memory : if true, DataLoader will copy tensors into pinned (page-locked) memory before returning them. Improves GPU transfer speed & overall training speed
*   drop_last : if true, DataLoader will drop last incomplete batch if total no. of samples is not divisible by batch_size
*   collate_fn : a callable that  processes a list of samples into a batch with custom logics
*   sampler : defines strategy for drawing samples, controls how batches are formed

# Improving the code for breast cancer data using mini batch gradient descent

In [26]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [27]:
data = pd.read_csv('/content/breast-cancer.csv')

In [28]:
Y = data['diagnosis']
X = data.drop(columns =['id','diagnosis'],axis=1)

In [29]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2)

In [30]:
scaler = StandardScaler()

In [31]:
X_train = scaler.fit_transform(X_train)
X_test = scaler.fit_transform(X_test)

In [32]:
encoder = LabelEncoder()

In [33]:
Y_train = encoder.fit_transform(Y_train)
Y_test = encoder.transform(Y_test)

In [38]:
X_train_tensor = torch.from_numpy(X_train)
X_train_tensor = X_train_tensor.float()
X_test_tensor = torch.from_numpy(X_test)
X_test_tensor = X_test_tensor.float()
Y_train_tensor = torch.from_numpy(Y_train)
Y_train_tensor = Y_train_tensor.float()
Y_test_tensor = torch.from_numpy(Y_test)
Y_test_tensor = Y_test_tensor.float()

**Updation**

In [39]:
from torch.utils.data import Dataset, DataLoader

class CustomDataset(Dataset):
  def __init__(self, features, labels):
    self.features = features
    self.labels = labels

  def __len__(self):
    return self.features.shape[0]

  def __getitem__(self, idx):
    return self.features[idx], self.labels[idx]

In [41]:
train_dataset = CustomDataset(X_train_tensor, Y_train_tensor)
test_dataset = CustomDataset(X_test_tensor, Y_test_tensor)

In [42]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)

In [43]:
import torch.nn as nn

class firstNN(nn.Module):
  def __init__(self, num_of_features):
    super().__init__()
    self.linear = nn.Linear(num_of_features, 1)
    self.sigmoid = nn.Sigmoid()

  def forward(self, features):
    out = self.linear(features)
    out = self.sigmoid(out)
    return out

In [44]:
learning_rate = 0.1
no_of_epochs = 40

In [46]:
model = firstNN(X_train_tensor.shape[1])

optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

loss_function = nn.BCELoss()

In [47]:
#define loop
for epochs in range(no_of_epochs):

  for batch_features, batch_labels in train_loader:

    #forward pass
    y_pred = model(X_train_tensor)

    #loss calculation
    loss = loss_function(y_pred, Y_train_tensor.view(-1,1))

    #zero gradients so that gradients doesn't accumulate
    optimizer.zero_grad()

    #backward pass
    loss.backward()

    #parameters updation
    optimizer.step()

    #visualizing loss with each epoch
    print(f'Epoch: {epochs+1}, Loss: {loss.item()}')

Epoch: 1, Loss: 0.5302042365074158
Epoch: 1, Loss: 0.4492335617542267
Epoch: 1, Loss: 0.3967719078063965
Epoch: 1, Loss: 0.3592921197414398
Epoch: 1, Loss: 0.3307865560054779
Epoch: 1, Loss: 0.3081609308719635
Epoch: 1, Loss: 0.2896396517753601
Epoch: 1, Loss: 0.2741202414035797
Epoch: 1, Loss: 0.26087701320648193
Epoch: 1, Loss: 0.2494095414876938
Epoch: 1, Loss: 0.23935995995998383
Epoch: 1, Loss: 0.23046454787254333
Epoch: 1, Loss: 0.22252382338047028
Epoch: 1, Loss: 0.21538372337818146
Epoch: 1, Loss: 0.20892302691936493
Epoch: 2, Loss: 0.20304463803768158
Epoch: 2, Loss: 0.1976698637008667
Epoch: 2, Loss: 0.1927339732646942
Epoch: 2, Loss: 0.18818311393260956
Epoch: 2, Loss: 0.18397215008735657
Epoch: 2, Loss: 0.18006281554698944
Epoch: 2, Loss: 0.17642246186733246
Epoch: 2, Loss: 0.17302298545837402
Epoch: 2, Loss: 0.16984011232852936
Epoch: 2, Loss: 0.16685277223587036
Epoch: 2, Loss: 0.16404256224632263
Epoch: 2, Loss: 0.16139324009418488
Epoch: 2, Loss: 0.15889059007167816
Epo

In [49]:
#model evaluation using test_loader
model.eval() # set model to evaluation mode
accuracy_list = []

with torch.no_grad():
  for batch_features, batch_labels in test_loader:
    y_pred = model(batch_features) #forward pass
    y_pred = (y_pred > 0.8).float() # convert probabilities to binary prediction

    # calculate accuracy for the current batch
    batch_accuracy = (y_pred.view(-1) == batch_labels).float().mean().item()
    accuracy_list.append(batch_accuracy)

#Calculate overall accuracy
overall_accuracy = sum(accuracy_list)/len(accuracy_list)
print(f'Accuracy: {overall_accuracy: .4f}')

Accuracy:  0.9392
